### TODO
---
- Define bits and bytes config
- Load the model and tokenizer, model `mistralai/Mistral-7B-v0.1`.
- Define LoRA config.
- Load the dataset, dataset name `wikitext-2-raw-v1`.
- Do the text preprocessing and tokenization.
- Define training arguments and trainer.
- Do the initial traing.
- Load the base model and make the final model.
- Now start instruction Fine-Tuning
    - Load instruct dataset `databricks/databricks-dolly-15k`
    - Formating the instructions and  Tokenize.
    - Define training args and trainer
    - Do the training 

In [1]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer , AutoModelForCausalLM, 
    TrainingArguments , Trainer,
    BitsAndBytesConfig , DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig , get_peft_model , prepare_model_for_kbit_training,
    TaskType , PeftModel
)
from trl import SFTTrainer , SFTConfig

In [2]:
MODEL_NAME = "google/gemma-2b"
PHASE1_OUTPUT_DIR = "./gemma2b-phase1-causalLM" # non-instruction fine-tuning
PHASE2_OUTPUT_DIR = "./gemma2b-phase2-instruct" # instruction fine-tuning
MAX_SEQ_LEN = 256
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
# 4-bit quantization config(QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_compute_dtype = torch.bfloat16,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_use_double_quant = True
)

In [4]:
# load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

In [5]:
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Model max length: {tokenizer.model_max_length}")
print(f"Pad token: '{tokenizer.pad_token}'")

Vocab size: 256000
Model max length: 1000000000000000019884624838656
Pad token: '<pad>'


In [6]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map = 'auto',
    trust_remote_code = True
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [7]:
model

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): Embedding(256000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x GemmaDecoderLayer(
        (self_attn): GemmaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=16384, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=16384, bias=False)
          (down_proj): Linear4bit(in_features=16384, out_features=2048, bias=False)
          (act_fn): GELUActivation()
        )
        (input_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
      )
    )
    (n

### Prepare Model for k-bit Training
  - `prepare_model_for_kbit_training()` does 3 things:
     1. Freeze all base model weights.
     2. Casts LayerNorm layers to FP32 for training stability
     3. Enables gradient checkpointing to reduce VRAM during backprop.

In [8]:
model = prepare_model_for_kbit_training(model)

### Define LoRA configuration

In [9]:
lora_config = LoraConfig(
    r = 16,
    lora_alpha = 32, # scaling = alpha/r
    lora_dropout = 0.05,
    bias = 'none',
    task_type = TaskType.CAUSAL_LM,
    target_modules=["k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model , lora_config)

In [10]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GemmaForCausalLM(
      (model): GemmaModel(
        (embed_tokens): Embedding(256000, 2048, padding_idx=0)
        (layers): ModuleList(
          (0-17): 18 x GemmaDecoderLayer(
            (self_attn): GemmaAttention(
              (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
              (k_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=256, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=256, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()


In [11]:
model.print_trainable_parameters()

trainable params: 17,252,352 || all params: 2,523,424,768 || trainable%: 0.6837


In [12]:
wiki_dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

In [13]:
wiki_dataset

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

In [14]:
print(wiki_dataset["train"][10]["text"][:300])

 The game 's battle system , the BliTZ system , is carried over directly from Valkyira Chronicles . During missions , players select each unit using a top @-@ down perspective of the battlefield map : once a character is selected , the player moves the character around the battlefield in third @-@ p


In [15]:
def tokenize(examples):
    result = tokenizer(
        examples['text'],
        truncation = True,
        max_length = MAX_SEQ_LEN,
        padding = 'max_length',
        return_overflowing_tokens = False 
    )
    return result

In [16]:
def group_texts(examples):
    """
    Concatenate all texts and chunk into blocks of MAX_SEQ_LEN.
    This avoids wasting tokens from padding short sequences.
    """
    # concatenate all sequences
    concatenated = {k: sum(examples[k] , []) for k in examples.keys()}
    # compute total length
    total_length = len(concatenated['input_ids'])
    # now let's say total length is more than MAX_SEQ_LEN then drop last incomplete chunk
    total_length = (total_length // MAX_SEQ_LEN) * MAX_SEQ_LEN
    # chunk into fixed blocks
    result = {
        k: [t[i : i + MAX_SEQ_LEN] for i in range(0, total_length, MAX_SEQ_LEN)]
        for k, t in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()

In [19]:
tokenize(
    wiki_dataset['train'][0]
)

{'input_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2], 'attention_mask': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [17]:
# Filter out empty/whitespace-only rows first
wiki_dataset = wiki_dataset.filter(lambda x: len(x["text"].strip()) > 10)

Filter:   0%|          | 0/4358 [00:00<?, ? examples/s]

Filter:   0%|          | 0/36718 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [18]:
tokenized_wiki = wiki_dataset.map(
    tokenize,
    batched = True,
    remove_columns = ["text"],
    desc = "Tokenizing WikiText-2",
)

Tokenizing WikiText-2:   0%|          | 0/2850 [00:00<?, ? examples/s]

Tokenizing WikiText-2:   0%|          | 0/23547 [00:00<?, ? examples/s]

Tokenizing WikiText-2:   0%|          | 0/2454 [00:00<?, ? examples/s]

In [19]:
lm_wiki = tokenized_wiki.map(
    group_texts,
    batched = True
)

Map:   0%|          | 0/2850 [00:00<?, ? examples/s]

Map:   0%|          | 0/23547 [00:00<?, ? examples/s]

Map:   0%|          | 0/2454 [00:00<?, ? examples/s]

In [21]:
# define training arguments for phase-01
phase1_training_args = TrainingArguments(
    num_train_epochs = 1,
    max_steps = -1, # run full epochs
    per_device_train_batch_size = 4,
    per_device_eval_batch_size = 4,
    gradient_accumulation_steps = 4,
    learning_rate = 2e-4,
    weight_decay = 0.01,
    lr_scheduler_type = 'cosine',
    warmup_ratio = 0.03,
    bf16 = True,
    gradient_checkpointing = True,
    optim = 'paged_adamw_32bit',
    logging_dir = "./logs/phase1",
    logging_steps = 50,
    eval_strategy = "steps",
    eval_steps = 200,
    save_steps = 200,
    save_total_limit = 2,
    load_best_model_at_end = True,
    metric_for_best_model = 'eval_loss',
    report_to = 'none'
)

In [22]:
# phase-01 training
data_collator = DataCollatorForLanguageModeling(
    tokenizer = tokenizer,
    mlm = False  
)

In [23]:
phase1_trainer = Trainer(
    model = model,
    args = phase1_training_args,
    train_dataset = lm_wiki["train"],
    eval_dataset = lm_wiki["validation"],
    tokenizer = tokenizer,
    data_collator = data_collator,
)

C:\Users\tipto\AppData\Local\Temp\ipykernel_19652\2941144137.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  phase1_trainer = Trainer(


In [24]:
phase1_result = phase1_trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss


KeyboardInterrupt: 